In [ ]:
import torch
if not torch.cuda.is_available():
  raise Exception("GPU not available")
device = torch.device("cuda")
print(f"Device: {device}")

Device: cuda


In [ ]:
import os
os.environ["GPTQMODEL_CPU_THREADS"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["NUM_PARLLEL_JOBS"] = "1"

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
import torch
import math
import time
import shutil
import psutil
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GPTQConfig,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

def get_process_memory_mb():
  """ Return how much RAM is the current program utilizing"""
  ## psutil --> Process and System Utilities
  process = psutil.Process(os.getpid()) ## os.getpid() --> Accessing the current process id!
  return process.memory_info().rss/1024 ** 2 ## rss --> Resident Set Size

def get_gpu_memory_mb():
  """Return current GPU memory in (MB) (if CUDA)"""
  if not torch.cuda.is_available():
    return 0.0
  return torch.cuda.memory_allocated() / 1024 ** 2

def describe_memory(label):
  cpu_mem = get_process_memory_mb()
  gpu_mem = get_gpu_memory_mb()
  print(f"[{label}] CPU memory: {cpu_mem:8.2f} MB | GPU memory: {gpu_mem:8.2f} MB")


@torch.no_grad()
def compute_perplexity(model, tokenizer,text: str) -> float:
  model.eval()
  enc = tokenizer(text, return_tensors="pt").to(device)
  outputs = model(**enc, labels=enc["input_ids"])
  return math.exp(outputs.loss.item())

@torch.no_grad()
def timed_generate(model,tokenizer,prompt:str,max_new_tokens: int = 40, num_runs: int = 3):
  model.eval()
  times = []
  last_output = True

  for i in range(num_runs):
    inputs = tokenizer(prompt,return_tensors = "pt").to(device)
    torch.cuda.empty_cache()
    start = time.perf_counter()
    out = model.generate(**inputs,max_new_tokens=max_new_tokens)
    end = time.perf_counter()
    times.append(end-start)
    last_output = tokenizer.decode(out[0], skip_special_tokens = True)
  avg_time = sum(times)/len(times)
  return avg_time,last_output

Device: cuda


In [ ]:
model_id = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

torch.cuda.empty_cache()
describe_memory("Before FP16 loading")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device)

describe_memory("After FP16 loading")
print("Baseline dtype:",next(model_fp16.parameters()).dtype)

[Before FP16 loading] CPU memory:   881.56 MB | GPU memory:     0.00 MB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[After FP16 loading] CPU memory:  1019.79 MB | GPU memory:   245.48 MB
Baseline dtype: torch.float16


In [ ]:
baseline_text = (
    "Quantization allows us to compress large language models like OPT while "
    "preserving most of their predictive power."
)
ppl_fp16 = compute_perplexity(model_fp16, tokenizer, baseline_text)
print(f"Baseline perplexity: {ppl_fp16:8.2f}")

prompt = "In the future, efficient Language Models will"
t_fp16,out_fp16 = timed_generate(model_fp16, tokenizer, prompt)

print(f"\n Baseline FP16 generation time: {t_fp16:.3f} s")
print("Baseline FP16 output:\n",out_fp16)

Baseline perplexity:   338.70

 Baseline FP16 generation time: 1.961 s
Baseline FP16 output:
 In the future, efficient Language Models will be used to create a new language model.

The language model will be used to create a new language model.

The language model will be used to create a new language model.



In [ ]:
calib_texts = [
    "Large language models can be quantized to 4 bits with minimal loss using GPTQ.",
    "GPTQ is a post-training, weight-only quantization method for transformer models.",
    "During quantization, GPTQ minimizes the error on the layer output, not only on the weights.",
    "Using second-order information, GPTQ adjusts the remaining weights to absorb quantization error.",
    "With 4-bit GPTQ, we can often fit much larger models on a single GPU while maintaining good quality.",
    "Group size and activation ordering are key hyperparameters that control the trade-off between speed and accuracy.",
]

In [ ]:
gptq_config = GPTQConfig(
    bits = 4, # 4 - bit weights
    tokenizer = tokenizer, # used to process the calibaration data
    dataset = calib_texts, # small custom calibration dataset
    group_size = 128, # Recommended Output
    damp_percent = 0.1, # Hessian Damping
    desc_act = True, # activation-order GPTQ (act-order)
    true_sequential = True,
    use_cuda_fp16 = True,
    pad_token_id = tokenizer.pad_token_id,
    device = device,
)
print(gptq_config)

[transformers] `act_group_aware` has been auto-disabled as it is not compatible with `desc_act = True`.


GPTQConfig(quant_method=<QuantizationMethod.GPTQ: 'gptq'>)


In [ ]:
!pip install optimum

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
describe_memory("Before GPTQ quantization")

start = time.perf_counter()
model_gptq = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
        max_memory={0: "30GiB", 1: "46GiB", "cpu": "30GiB"},
    quantization_config=gptq_config,
)

end = time.perf_counter()

print(f"\nGPTQ quantization time: {412} seconds")

[Before GPTQ quantization] CPU memory:  1515.49 MB | GPU memory:   255.23 MB


NameError: name 'QuantizeConfig' is not defined